In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Ustawianie poziomu optymalizacji Transpilera

<!-- source hash: d528e2b0 -->




<details>
<summary><b>Wersje pakietów</b></summary>

Kod na tej stronie został opracowany przy użyciu poniższych wymagań.
Zalecamy używanie tych wersji lub nowszych.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Prawdziwe urządzenia kwantowe są podatne na szum i błędy bramek, dlatego optymalizacja Circuit w celu zmniejszenia ich głębokości i liczby bramek może znacząco poprawić wyniki uzyskiwane z wykonania tych Circuit.
Funkcja [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) ma jeden wymagany argument pozycyjny, `optimization_level`, który kontroluje, ile wysiłku Transpiler wkłada w optymalizację Circuit. Ten argument może być liczbą całkowitą przyjmującą jedną z wartości 0, 1, 2 lub 3.
Wyższe poziomy optymalizacji generują bardziej zoptymalizowane Circuit kosztem dłuższego czasu kompilacji.
Poniższa tabela wyjaśnia optymalizacje przeprowadzane przy każdym ustawieniu.

<table>
  <thead>
    <tr>
      <th>Poziom optymalizacji</th>
      <th>Opis</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>0</td>
      <td>
        Brak optymalizacji: zazwyczaj używany do charakteryzacji sprzętu
        - Podstawowe tłumaczenie
        - Układ/Routing: `TrivialLayout`, gdzie wybiera te same numery fizycznych Qubitów co wirtualne i wstawia SWAP, aby to działało (przy użyciu `SabreSwap`)
      </td>
    </tr>
    <tr>
      <td>1</td>
      <td>
        Lekka optymalizacja:
        -   Układ/Routing: Najpierw próbowany jest układ z `TrivialLayout`. Jeśli wymagane są dodatkowe SWAP, układ z minimalną liczbą SWAP jest znajdowany za pomocą `SabreSwap`, następnie używany jest `VF2LayoutPostLayout`, aby wybrać najlepsze Qubity w grafie.
        -   `InverseCancellation`
        -   Optymalizacja bramek 1Q
      </td>
    </tr>
    <tr>
      <td>2</td>
      <td>
        Średnia optymalizacja:
          - Układ/Routing: Poziom optymalizacji 1 (bez trivial) + heurystyczna optymalizacja z większą głębokością przeszukiwania i próbami funkcji optymalizacji. Ponieważ `TrivialLayout` nie jest używany, nie ma próby użycia tych samych numerów fizycznych i wirtualnych Qubitów.
        -   `CommutativeCancellation`
      </td>
    </tr>
    <tr>
      <td>3</td>
      <td>
        Wysoka optymalizacja:
        - Poziom optymalizacji 2 + heurystyczna optymalizacja układu/routingu z większym wysiłkiem/próbami
        - Resynteza bloków dwuqubitowych przy użyciu [dekompozycji KAK Cartana](https://arxiv.org/abs/quant-ph/0507171).
        - Przebiegi łamiące unitarność:
          * `OptimizeSwapBeforeMeasure`: Przesuwa pomiary, aby unikać SWAP
          * `RemoveDiagonalGatesBeforeMeasure`: Usuwa bramki przed pomiarami, które nie wpływałyby na pomiary
      </td>
    </tr>
  </tbody>
</table>

## Poziom optymalizacji w działaniu
Ponieważ bramki dwuqubitowe są zazwyczaj najważniejszym źródłem błędów, możemy w przybliżeniu określić ilościowo „efektywność sprzętową" transpilacji, licząc liczbę bramek dwuqubitowych w wynikowym Circuit.
Tutaj wypróbujemy różne poziomy optymalizacji na wejściowym Circuit składającym się z losowej unitarnej, po której następuje bramka SWAP.

In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import Operator, random_unitary

UU = random_unitary(4, seed=12345)
rand_U = UnitaryGate(UU)

qc = QuantumCircuit(2)
qc.append(rand_U, range(2))
qc.swap(0, 1)
qc.draw("mpl", style="iqp")

<Image src="../docs/images/guides/set-optimization/extracted-outputs/81173ebc-8359-48a6-b585-0477907b3b93-0.svg" alt="Output of the previous code cell" />

![Wynik poprzedniej komórki kodu](../docs/images/guides/set-optimization/extracted-outputs/81173ebc-8359-48a6-b585-0477907b3b93-0.svg)

Użyjemy atrapowego Backend `FakeSherbrooke` w naszych przykładach. Najpierw transpilujmy przy użyciu poziomu optymalizacji 0.

In [2]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()

pass_manager = generate_preset_pass_manager(
    optimization_level=0, backend=backend, seed_transpiler=12345
)
qc_t1_exact = pass_manager.run(qc)
qc_t1_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/40cdd173-b437-48b1-8928-741e8411342e-0.svg" alt="Output of the previous code cell" />

![Wynik poprzedniej komórki kodu](../docs/images/guides/set-optimization/extracted-outputs/40cdd173-b437-48b1-8928-741e8411342e-0.svg)

Transpilowany Circuit ma sześć dwuqubitowych bramek ECR.

Powtórzmy dla poziomu optymalizacji 1:

In [3]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()

pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend, seed_transpiler=12345
)
qc_t1_exact = pass_manager.run(qc)
qc_t1_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/2dab5def-a017-42e9-92d6-e043ac4065b2-0.svg" alt="Output of the previous code cell" />

![Wynik poprzedniej komórki kodu](../docs/images/guides/set-optimization/extracted-outputs/2dab5def-a017-42e9-92d6-e043ac4065b2-0.svg)

Transpilowany Circuit nadal ma sześć bramek ECR, ale liczba jednoqubitowych bramek się zmniejszyła.

Powtórzmy dla poziomu optymalizacji 2:

In [4]:
pass_manager = generate_preset_pass_manager(
    optimization_level=2, backend=backend, seed_transpiler=12345
)
qc_t2_exact = pass_manager.run(qc)
qc_t2_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/77d76048-b1e8-4225-b35f-80dc9d458e8d-0.svg" alt="Output of the previous code cell" />

![Wynik poprzedniej komórki kodu](../docs/images/guides/set-optimization/extracted-outputs/77d76048-b1e8-4225-b35f-80dc9d458e8d-0.svg)

To daje te same wyniki co poziom optymalizacji 1. Pamiętaj, że zwiększenie poziomu optymalizacji nie zawsze robi różnicę.

Powtórzmy ponownie z poziomem optymalizacji 3:

In [5]:
pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend, seed_transpiler=12345
)
qc_t3_exact = pass_manager.run(qc)
qc_t3_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/4109d0e2-df37-4850-8409-6b860c48595c-0.svg" alt="Output of the previous code cell" />

![Wynik poprzedniej komórki kodu](../docs/images/guides/set-optimization/extracted-outputs/4109d0e2-df37-4850-8409-6b860c48595c-0.svg)

Teraz są tylko trzy bramki ECR. Osiągamy ten wynik, ponieważ na poziomie optymalizacji 3 Qiskit próbuje powtórnie syntetyzować dwuqubitowe bloki bramek, a każda dwuqubitowa bramka może być zaimplementowana przy użyciu co najwyżej trzech bramek ECR. Możemy uzyskać jeszcze mniej bramek ECR, jeśli ustawimy `approximation_degree` na wartość mniejszą niż 1, pozwalając Transpilerowi na przybliżenia, które mogą wprowadzić pewien błąd w dekompozycji bramki (patrz [Często używane parametry transpilacji](common-parameters#approximation-degree)):

In [6]:
pass_manager = generate_preset_pass_manager(
    optimization_level=3,
    approximation_degree=0.99,
    backend=backend,
    seed_transpiler=12345,
)
qc_t3_approx = pass_manager.run(qc)
qc_t3_approx.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/bf239116-b8bb-42aa-a27a-89206d9e108a-0.svg" alt="Output of the previous code cell" />

![Wynik poprzedniej komórki kodu](../docs/images/guides/set-optimization/extracted-outputs/bf239116-b8bb-42aa-a27a-89206d9e108a-0.svg)

Ten Circuit ma tylko dwie bramki ECR, ale jest przybliżonym Circuit. Aby zrozumieć, czym różni się jego działanie od dokładnego Circuit, możemy obliczyć wierność między operatorem unitarnym, który implementuje ten Circuit, a dokładnym unitarnym. Przed wykonaniem obliczeń najpierw redukujemy transpilowany Circuit, który zawiera 127 Qubitów, do Circuit zawierającego tylko aktywne Qubity, których jest dwa.

In [7]:
import numpy as np


def trace_to_fidelity_2q(trace: float) -> float:
    return (4.0 + trace * trace.conjugate()) / 20.0


# Reduce circuits down to 2 qubits so they are easy to simulate
qc_t3_exact_small = QuantumCircuit.from_instructions(qc_t3_exact)
qc_t3_approx_small = QuantumCircuit.from_instructions(qc_t3_approx)

# Compute the fidelity
exact_fid = trace_to_fidelity_2q(
    np.trace(np.dot(Operator(qc_t3_exact_small).adjoint().data, UU))
)
approx_fid = trace_to_fidelity_2q(
    np.trace(np.dot(Operator(qc_t3_approx_small).adjoint().data, UU))
)
print(
    f"Synthesis fidelity\nExact: {exact_fid:.3f}\nApproximate: {approx_fid:.3f}"
)

Synthesis fidelity
Exact: 1.000+0.000j
Approximate: 0.992+0.000j


Adjusting the optimization level can change other aspects of the circuit too, not just the number of ECR gates. For examples of how setting optimization level changes the layout, see [Representing quantum computers](./represent-quantum-computers).

## Next steps

<Admonition type="tip" title="Recommendations">
    - To learn more about the `generate_preset_passmanager` function, start with the [Transpilation default settings and configuration options](defaults-and-configuration-options) topic.
    - Continue learning about transpilation with the [Transpiler stages](transpiler-stages) topic.
    - Try the [Compare transpiler settings](/docs/guides/circuit-transpilation-settings) guide.
    - Try the [Build repetition codes](/docs/tutorials/repetition-codes) tutorial.
    - See the [Transpile API documentation.](/docs/api/qiskit/transpiler)
</Admonition>